# 🎙️ Proyecto Auditoria DIGI — Fase 2: Transcripción

**Nodos cubiertos:** 5. faster-whisper → 8. stable-ts (alineación) → 9. pyannote (diarización)

**Entrada:** `audio_vad.wav` de Fase 1  
**Salida:** guión diarizado listo para Fase 3

> **Colab vs Docker:** Usamos `faster-whisper` + `stable-ts` en Colab (sin conflictos de NumPy). En Docker final se usa `whisperX` completo — el resultado es idéntico.

---
**Orden obligatorio:** ejecuta las celdas de arriba a abajo.

## 🔧 Paso 0 — Auto-detección de entorno
Esta celda detecta la versión de NumPy y la corrige si es necesario. **Ejecuta siempre primero.**

In [ ]:
import subprocess, sys
import importlib.util

# ─── Auto-detección y fix de NumPy ───────────────────────────
spec = importlib.util.find_spec("numpy")
if spec:
    import numpy as _np
    version = tuple(int(x) for x in _np.__version__.split(".")[:2])
    if version >= (2, 0):
        print(f"⚠️  NumPy {_np.__version__} detectado — incompatible con pyannote")
        print("   Instalando NumPy 1.26.4 sobre el sistema...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "numpy==1.26.4",
            "--target", "/usr/local/lib/python3.12/dist-packages/",
            "--no-deps", "-q"
        ])
        print("✅ NumPy 1.26.4 instalado")
        print("")
        print("🔴 ACCIÓN REQUERIDA: Runtime → Restart session")
        print("   Después del reinicio ejecuta todo desde esta celda — ya no volverá a pedir reinicio")
    else:
        print(f"✅ NumPy {_np.__version__} — compatible, continúa")
else:
    print("NumPy no instalado aún — continúa con Paso 1")


## ✅ Paso 0 — Verificar GPU y NumPy

In [ ]:
import subprocess, numpy as np

r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ GPU disponible:')
    for linea in r.stdout.split('\n'):
        if any(x in linea for x in ['Tesla','RTX','A100','T4','V100','L4']):
            print(' ', linea.strip()); break
else:
    print('❌ Sin GPU')

print(f'\nNumPy versión: {np.__version__} (cualquier versión es compatible con este notebook)')

## 📦 Paso 1 — Instalar dependencias
Tarda ~2 minutos. Compatible con cualquier versión de NumPy. No requiere reinicio.

In [ ]:
# Motor ASR (el mismo que usa WhisperX internamente)
!pip install -q faster-whisper
print('  ✓ faster-whisper')

# Alineación de palabras con timestamps

# Diarización — quién habla cuándo
!pip install -q pyannote.audio
print('  ✓ pyannote.audio')

# Audio utils
!pip install -q torchaudio soundfile
print('  ✓ torchaudio + soundfile')

print('\n✅ Todo instalado — continúa con el Paso 2')

## 🔑 Paso 2 — Token de HuggingFace
Solo vive en memoria durante esta sesión — nunca se guarda en el código.

In [ ]:
import getpass

HF_TOKEN = getpass.getpass('🔑 Token HuggingFace (hf_...): ')

if HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 20:
    print('✅ Token recibido')
else:
    print('⚠️ Comprueba que empieza por hf_')

## 🎵 Paso 3 — Cargar audio de Fase 1

In [ ]:
import os

ARCHIVO_AUDIO = 'audio_vad.wav'

if os.path.exists(ARCHIVO_AUDIO):
    print(f'✅ audio_vad.wav encontrado ({os.path.getsize(ARCHIVO_AUDIO)/1024:.0f} KB)')
else:
    print('📂 No encontrado — sube el audio_vad.wav de Fase 1:')
    from google.colab import files
    subidos = files.upload()
    ARCHIVO_AUDIO = list(subidos.keys())[0]
    print(f'✅ Recibido: {ARCHIVO_AUDIO}')

## 🎤 Nodo 5 — faster-whisper: Transcripción

**¿Qué hace?** Convierte el audio en texto. Mismo motor que WhisperX, compatible con NumPy 2.x. En Docker usaremos Whisper Large V3.

In [ ]:
from faster_whisper import WhisperModel
import gc, torch

IDIOMA = 'es'

print('🎤 Cargando faster-whisper Medium...')
modelo_asr = WhisperModel('medium', device='cuda', compute_type='int8')

print('   Transcribiendo... (~2-4 min para 20 min de audio)')
segmentos_gen, info = modelo_asr.transcribe(
    ARCHIVO_AUDIO,
    language=IDIOMA,
    beam_size=5,
    word_timestamps=True   # necesario para alineación posterior
)

# Materializar el generador
segmentos = list(segmentos_gen)

del modelo_asr; gc.collect(); torch.cuda.empty_cache()

# Construir estructura compatible con el resto del pipeline
texto_total = ' '.join([s.text.strip() for s in segmentos])
n_segmentos = len(segmentos)
n_palabras = len(texto_total.split())

print(f'\n✅ Nodo 5 completado')
print(f'   Segmentos: {n_segmentos} | Palabras: {n_palabras}')
print(f'   Idioma detectado: {info.language} (probabilidad: {info.language_probability:.0%})')
print(f'\n   Vista previa:')
print(f'   {texto_total[:300]}...')

## ⏱️ Nodo 8 — stable-ts: Alineación por palabra

**¿Qué hace?** Asigna timestamps exactos a cada palabra para saber cuándo habla cada persona.

In [ ]:
print("⏱️ Nodo 8 — Alineación por palabra")
# faster-whisper ya entrega word_timestamps directamente — no necesitamos stable-ts

segmentos_alineados = []
palabras_total = 0

for seg in segmentos:
    palabras = []
    if seg.words:
        for w in seg.words:
            palabras.append({
                'word': w.word,
                'start': w.start,
                'end': w.end,
                'score': w.probability
            })
        palabras_total += len(palabras)

    segmentos_alineados.append({
        'start': seg.start,
        'end': seg.end,
        'text': seg.text.strip(),
        'words': palabras
    })

print(f"✅ Nodo 8 completado")
print(f"   Palabras con timestamp: {palabras_total}")
if segmentos_alineados:
    s = segmentos_alineados[0]
    print(f'   Ejemplo: [{s["start"]:.1f}s→{s["end"]:.1f}s] "{s["text"]}"')


## 👥 Nodo 9 — pyannote: Diarización

**¿Qué hace?** Separa automáticamente agente y cliente — asigna cada fragmento a SPEAKER_00, SPEAKER_01...

In [ ]:
from pyannote.audio import Pipeline
import torch

print('👥 Cargando pyannote speaker-diarization-3.1...')
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    token=HF_TOKEN
)
pipeline = pipeline.to(torch.device('cuda'))

print('   Separando interlocutores... (~1-2 min)')
diarizacion = pipeline(ARCHIVO_AUDIO)

# Convertir a lista de segmentos con speaker
turnos = []
for turno, _, speaker in diarizacion.itertracks(yield_label=True):
    turnos.append({'start': turno.start, 'end': turno.end, 'speaker': speaker})

# Contar por speaker
speakers = {}
for t in turnos:
    speakers[t['speaker']] = speakers.get(t['speaker'], 0) + 1

print(f'\n✅ Nodo 9 completado')
print(f'   Interlocutores detectados: {len(speakers)}')
for sp, n in sorted(speakers.items()):
    print(f'   {sp}: {n} turnos')

## 📝 Asignar speaker a cada segmento y generar guión

In [ ]:
import json

def asignar_speaker(seg_start, seg_end, turnos):
    """Devuelve el speaker con mayor solapamiento con el segmento."""
    max_overlap = 0
    speaker_asignado = 'DESCONOCIDO'
    for t in turnos:
        overlap = min(seg_end, t['end']) - max(seg_start, t['start'])
        if overlap > max_overlap:
            max_overlap = overlap
            speaker_asignado = t['speaker']
    return speaker_asignado

# Asignar speaker a cada segmento de texto
guion_lineas = []
speaker_anterior = None
bloque_actual = ''

for seg in segmentos_alineados:
    speaker = asignar_speaker(seg['start'], seg['end'], turnos)
    texto = seg['text'].strip()

    if speaker != speaker_anterior:
        if bloque_actual:
            guion_lineas.append({'speaker': speaker_anterior, 'texto': bloque_actual.strip()})
        bloque_actual = texto + ' '
        speaker_anterior = speaker
    else:
        bloque_actual += texto + ' '

if bloque_actual:
    guion_lineas.append({'speaker': speaker_anterior, 'texto': bloque_actual.strip()})

# Vista previa
print('📝 Vista previa (primeras 15 intervenciones):')
print('=' * 60)
for item in guion_lineas[:15]:
    print(f"[{item['speaker']}]: {item['texto'][:120]}")
print('=' * 60)
print(f'Total intervenciones: {len(guion_lineas)}')

# Guardar guión y metadatos
GUION_TEXTO = '\n'.join([f"[{l['speaker']}]: {l['texto']}" for l in guion_lineas])

with open('guion_diarizado.txt', 'w', encoding='utf-8') as f:
    f.write(GUION_TEXTO)

resultado_fase2 = {
    'archivo_audio': ARCHIVO_AUDIO,
    'n_segmentos': n_segmentos,
    'n_palabras': n_palabras,
    'n_interlocutores': len(speakers),
    'turnos_por_speaker': speakers,
    'n_intervenciones_guion': len(guion_lineas),
    'archivo_guion': 'guion_diarizado.txt'
}

with open('resultado_fase2.json', 'w', encoding='utf-8') as f:
    json.dump(resultado_fase2, f, indent=2, ensure_ascii=False)

print('\n✅ Archivos guardados:')
print('   guion_diarizado.txt   → Fase 3 (análisis IA)')
print('   resultado_fase2.json  → metadatos')
print(f'\n➡️  Listo para Fase 3: Qwen3-8B + rúbrica DIGI')